In [1]:
import polars as pl


## Length

In [2]:
lenData = pl.read_parquet("/dataHDD1/pmData/s2orcParsed/len.parquet")
locData = pl.read_parquet("/dataHDD1/pmData/s2orcParsed/loc.parquet")
typeData = pl.read_parquet("/dataHDD1/pmData/s2orcParsed/type.parquet")

lenIds = lenData["corpusid"].to_list()
locIds = locData["corpusid"].to_list()
typeIds = typeData["corpusid"].to_list()

lenList = []
locList = []
typeList = []

for i in range(30):
    text = pl.read_parquet("/dataHDD1/pmData/news2cDownloads/" + str(i) + ".parquet")
    # filtered = text.filter(pl.col("corpusid").is_in(corpIds))
    lenJoined = lenData.join(text, on="corpusid")
    lenList.append(lenJoined)
    locJoined = locData.join(text, on="corpusid")
    locList.append(locJoined)
    typeJoined = typeData.join(text, on="corpusid")
    typeList.append(typeJoined)


In [3]:
lenRes = []
locRes = []
typeRes = []

for x in lenList:
    x = x.drop(['content.source.oainfo', 'content.source.oainfo.license', 'content.source.oainfo.openaccessurl', 'content.source.oainfo.status'])
    lenRes.append(x)
    
for x in locList:
    x = x.drop(['content.source.oainfo', 'content.source.oainfo.license', 'content.source.oainfo.openaccessurl', 'content.source.oainfo.status'])
    locRes.append(x)
    
for x in typeList:
    x = x.drop(['content.source.oainfo', 'content.source.oainfo.license', 'content.source.oainfo.openaccessurl', 'content.source.oainfo.status'])
    typeRes.append(x)
    
lenConcat = pl.concat(lenRes)    
locConcat = pl.concat(locRes)    
typeConcat = pl.concat(typeRes)    


In [4]:
lenConcat.write_parquet("/dataHDD1/pmData/selected/comLen.parquet")
locConcat.write_parquet("/dataHDD1/pmData/selected/comloc.parquet")
typeConcat.write_parquet("/dataHDD1/pmData/selected/comType.parquet")

In [8]:
data = """29
1050
============
27
1050
============
29
1050
============
35
1050
============
33
1050
============
37
1050
============
32
1050
============
28
1050
============
29
1050
============
30
1050
============
36
1050
============
42
1050
============
34
1050
============
46
1050
============
32
1050
============
32
1050
============
30
1050
============
26
1050
============
47
1050
============
37
1050
============
42
1050
============
41
1050
============
32
1050
============
44
1050
============
52
1050
============
33
1050
============
30
1050
============
34
1050
============
38
1050
============
33
1050
============"""

# Split the data into lines
lines = data.split("\n")

# Remove lines containing "1050" and "============"
filtered_lines = [line for line in lines if "1050" not in line and "============" not in line]

# Extract numbers and calculate the sum
sum_of_numbers = sum(int(line) for line in filtered_lines if line.strip().isdigit())

print("Sum of the remaining numbers:", sum_of_numbers)


Sum of the remaining numbers: 1050


In [10]:
import pandas as pd
df = pd.read_parquet("/dataHDD1/pmData/selected/comloc.parquet")
len(df)

3500

In [11]:
import sys
sys.path.append("/home/pop533611/thesis")
import pandas as pd
from pymongo import MongoClient
import mongoConnect as mc

df = pd.read_parquet("/dataHDD1/pmData/selected/comloc.parquet")
db = mc.connect()
df.columns
df = df.drop("content.source.pdfurls", axis=1)
data_records = df.to_dict(orient='records')

mc.save_many(db,"citedLoc",data_records)

hello


In [12]:
import sys
sys.path.append("/home/pop533611/thesis")
import pandas as pd
from pymongo import MongoClient
import mongoConnect as mc
db = mc.connect()
from API_request import APIRequest
api = APIRequest()

records = mc.read_records(db,"citedLoc")
dataList = []
for document in records:
    if "content.source.pdfsha" in document and document["content.source.pdfsha"]:
        doc = api.getCitingData(document["content.source.pdfsha"])
        if "data" in doc:
            data = doc["data"]
            for d in data:
                if d["citingPaper"]["isOpenAccess"] and d["citingPaper"]["year"] and d["citingPaper"]["title"] and d["citingPaper"]["publicationDate"] and d["citingPaper"]["journal"] and d["citingPaper"]["authors"]:
                    obj = {
                        "citingCID":d["citingPaper"]["corpusId"], 
                        "citingId":d["citingPaper"]["paperId"], 
                        "meta": d, 
                        "citedPaperId":document["content.source.pdfsha"], 
                        "citedPaperCId": document["corpusid"],
                        "citingPaper": document["_id"]}
                    dataList.append(obj)
                    mc.save_record(db, "citingLoc", obj)
                
            
            

In [13]:
import sys
sys.path.append("/home/pop533611/thesis")
import pandas as pd
from pymongo import MongoClient
import mongoConnect as mc
db = mc.connect()
from API_request import APIRequest
api = APIRequest()

import polars as pl
df = pl.DataFrame(mc.read_records(db,"citingLoc"))
df.columns
citingJoined = []
for i in range(30):
    print("=================================")
    print(i)
    text = pl.read_parquet("/dataHDD1/pmData/news2cDownloads/" + str(i) + ".parquet")
    # filtered = text.filter(pl.col("corpusid").is_in(corpIds))
    citingJoined.append(df.join(text, right_on="corpusid", left_on="citingCID"))
    
    
    

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29


In [14]:
citingList = []
for x in citingJoined:
    x = x.drop(['content.source.oainfo','meta', 'content.source.pdfurls','content.source.oainfo.license', 'content.source.oainfo.openaccessurl', 'content.source.oainfo.status'])
    citingList.append(x)
    
citingConcat = pl.concat(citingList)

In [15]:
citingConcat.columns

['_id',
 'citingCID',
 'citingId',
 'citedPaperId',
 'citedPaperCId',
 'citingPaper',
 'updated',
 'externalids.arxiv',
 'externalids.mag',
 'externalids.acl',
 'externalids.pubmed',
 'externalids.pubmedcentral',
 'externalids.dblp',
 'externalids.doi',
 'content.source.pdfurls',
 'content.source.pdfsha',
 'content.text',
 'content.annotations.abstract',
 'content.annotations.author',
 'content.annotations.authoraffiliation',
 'content.annotations.authorfirstname',
 'content.annotations.authorlastname',
 'content.annotations.bibauthor',
 'content.annotations.bibauthorfirstname',
 'content.annotations.bibauthorlastname',
 'content.annotations.bibentry',
 'content.annotations.bibref',
 'content.annotations.bibtitle',
 'content.annotations.bibvenue',
 'content.annotations.figure',
 'content.annotations.figurecaption',
 'content.annotations.figureref',
 'content.annotations.formula',
 'content.annotations.paragraph',
 'content.annotations.publisher',
 'content.annotations.sectionheader',
 

In [15]:
import uuid

citingConcatPd = citingConcat.to_pandas()

citingConcatPd['_id'] = [str(uuid.uuid4())[:24] for _ in range(len(citingConcatPd))]


citingConcatRc = citingConcatPd.to_dict(orient = "records")
# col = db["citingTextType"]

# col.insert_many(citingConcatRc)
mc.save_many(db,"citingTextLoc",citingConcatRc)

hello


In [2]:
citingList = []
for x in citingJoined:
    x = x.drop(['content.source.oainfo', 'content.source.oainfo.license', 'content.source.oainfo.openaccessurl', 'content.source.oainfo.status'])
    citingList.append(x)
    
citingConcat = pl.concat(citingList)
citingConcat
# citingConcat.write_parquet("/dataHDD1/pmData/selected/citingText.parquet")
# citingPd = citingConcat.to_pandas()
# citingConcatRecords = citingPd.to_dict(orient='records')
# print(citingConcatRecords)

# for record in citingConcatRecords:
#     record['content.text'] = ' '.join(record['content.text'])
#     print(record["citedPaperCId"])
#     mc.save_record(db,"citingLenText",record)

NameError: name 'citingJoined' is not defined

In [39]:
import sys
import time
import json
sys.path.append("/home/pop533611/thesis")
from API_request import APIRequest
api = APIRequest()

from random import randrange
data = pl.read_parquet("/dataHDD1/pmData/selected/comLen.parquet")
res = []


grouped = data.groupby('category')
for cat, grp in grouped:
    for i in [(1996,2000),(2001,2005),(2006,2010),(2011,2015),(2016,2020)]:
        for j in range(10):
            grp_test = grp.filter(
                (pl.col("year").is_in(list(range(i[0], i[1] + 1)))) &
                (pl.col("group") == j)
            ) 
            
            for g in grp_test:
                print(g)
                # doc = api.getCitingData(idx)
                # data = doc["data"]
                # for d in data:
                #     if d["citingPaper"]["isOpenAccess"]:
                #         obj = {"category": cat, "year":grp_test["year"],"group": j, "corpusid": grp_test }
                        

            
            
# for idx in data["content.source.pdfsha"]:
#     doc = api.getCitingData(idx)
    
#     data = doc["data"]
    
#     print("data: " , len(data))
#     t = []
#     for d in data:
#         print(d["citingPaper"]["isOpenAccess"])
#         if d["citingPaper"]["isOpenAccess"]:
#             t.append(d["citingPaper"]["paperId"])
    
#     if(len(t) == 0):
#         res.append(None)
#     else:
#         print(len(t))
#         selData = randrange(len(t))
#         res.append(t[selData])

# print(res)
    
    

    
    


shape: (10,)
Series: 'corpusid' [i64]
[
	119063254
	119661137
	2841442
	8230092
	15882948
	118868893
	14834489
	17461548
	3040559
	2332239
]
shape: (10,)
Series: 'year' [i32]
[
	2000
	2000
	1998
	1999
	2000
	1998
	1997
	1998
	1998
	1997
]
shape: (10,)
Series: 'citationcount' [i64]
[
	40
	24
	12
	18
	40
	14
	15
	12
	21
	44
]
shape: (10,)
Series: 'category' [str]
[
	"Mathematics"
	"Mathematics"
	"Mathematics"
	"Mathematics"
	"Mathematics"
	"Mathematics"
	"Mathematics"
	"Mathematics"
	"Mathematics"
	"Mathematics"
]
shape: (10,)
Series: 'group' [i32]
[
	0
	0
	0
	0
	0
	0
	0
	0
	0
	0
]
shape: (10,)
Series: 'updated' [str]
[
	"2022-09-18T03:…
	"2022-09-24T02:…
	"2022-07-05T20:…
	"2022-10-11T03:…
	"2022-07-23T10:…
	"2022-09-15T00:…
	"2022-09-11T00:…
	"2022-09-24T23:…
	"2022-07-13T09:…
	"2022-07-13T07:…
]
shape: (10,)
Series: 'externalids.arxiv' [str]
[
	"nlin/0005061"
	"math/0007158"
	"cs/9809015"
	"math-ph/990803…
	"math/0010037"
	"hep-th/9801094…
	"q-alg/9710028"
	"math/9810194"
	"math/98021

In [16]:
papers = pl.read_parquet("/dataHDD1/pmData/s2orcParsed/combinedPapers.parquet")

NameError: name 'pl' is not defined

In [2]:
import json
import numpy as np
from scipy.stats import skew, kurtosis
import csv
import pandas as pd

df_exploded = papers.explode('s2fieldsofstudy')
exploded_list = [i for i in df_exploded["s2fieldsofstudy"]]
df_sep = pl.from_dicts(exploded_list)
df_res = df_exploded.with_columns(df_sep)
df_filter = df_res.filter((pl.col("source") == "external") & (pl.col("isopenaccess") == True))
df_filter = df_filter.with_columns(
    pl.col("year").cast(pl.Int32).alias("year")
)

print("len: ", len(df_filter))
grouped = df_filter.groupby('category')
newDF = []
fos = []
grp = []

records = []
results = []

csv_list = []

df = pd.DataFrame()
catList = []
yearList= []
ccListDF = []

df_filter = df_filter.select(["corpusid", "year", "citationcount","category"])
df_filter = df_filter.with_columns(pl.lit(None).alias("group"))

filter_list = []

yearsCount = 0
cCount = 0
c = 0
for cat, group in grouped:
    print(cat)
    c += len(group)
    
    per = (len(group)/6968058) * 100
    # if per >= 3:
    for i in [(1996,2000),(2001,2005),(2006,2010),(2011,2015),(2016,2020)]:

        group_filter = group.filter(pl.col("year").is_in(list(range(i[0],i[1] + 1))))
        print("----------------------")
        yearsCount += len(group_filter)
        
            
        ccList = group_filter["citationcount"].to_list()
        cc = sorted(set(ccList))
            
        if len(cc) > 3:
            
            elements_per_group = int(len(cc) / 3)

                
            groups = [cc[i:i+elements_per_group] for i in range(0, len(cc), elements_per_group)]
            for ind,j in enumerate(groups[:3]):
                catList.append(cat)
                yearList.append(str(i))
                ccListDF.append(str(j))
            
                df_filter_1 = df_filter.filter(
                    (df_filter["category"] == cat) &
                    df_filter["year"].is_in(list(range(i[0], i[1] + 1))) &
                    df_filter["citationcount"].is_in(j)
                ).with_columns(
                        pl.lit(ind).alias("group")
                    )
                cCount += len(df_filter_1)
                filter_list.append(df_filter_1)
                
df["field"] = catList
df["year"] = yearList
df["cc"] = ccListDF
concatenated_df = pl.concat(filter_list)
concatenated_df.write_parquet("/dataHDD1/pmData/s2orcParsed/filteredGroup.parquet")
print("total: ", c)
print(len(concatenated_df))
print(yearsCount)
print(cCount)


NameError: name 'papers' is not defined

In [ ]:
df = pl.read_parquet("/dataHDD1/pmData/s2orcParsed/filteredGroup.parquet")
grouped = df.groupby('category')

In [1]:
import random

final = []
lenData = []
locData = []
typeData = []

c = 0
for cat, grp in grouped:
    print(cat)

    for i in [(1996,2000),(2001,2005),(2006,2010),(2011,2015),(2016,2020)]:
        for j in range(3):
            temp_cid = []
            grp_test = grp.filter(
                (pl.col("year").is_in(list(range(i[0], i[1] + 1)))) &
                (pl.col("group") == j) & 
                (pl.col("citationcount") >= 5)
            ) 
            grp_test_list = grp_test["corpusid"].to_list()
            
            print(str(i[0]) + "-" + str(i[1] + 1) + ":"  + str(j) + ":" + str(len(grp_test_list)))
            
            random.shuffle(grp_test_list)
                                  
            filtered_df = grp_test.filter(pl.col("corpusid").is_in(grp_test_list[:3]))[:3]
            lenData.append(filtered_df)
            
            filtered_df = grp_test.filter(pl.col("corpusid").is_in(grp_test_list[3:6]))[:3]
            locData.append(filtered_df)
            
            filtered_df = grp_test.filter(pl.col("corpusid").is_in(grp_test_list[6:9]))[:3]
            typeData.append(filtered_df)
            

    print("=================================================")

            

           
lenConcat = pl.concat(lenData)
locConcat = pl.concat(locData)
typeConcat = pl.concat(typeData)



NameError: name 'grouped' is not defined

In [ ]:
import sys
sys.path.append("/home/pop533611/thesis")
import pandas as pd
from pymongo import MongoClient
import mongoConnect as mc

In [ ]:
ids = []

In [ ]:
data = mc.read_records("citedLoc")
for d in data:
    print(d)
    ids.append(d["corpusid"])
data = mc.read_records("citedLen")
for d in data:
    ids.append(d["corpusid"])
data = mc.read_records("citedType")
for d in data:
    ids.append(d["corpusid"])

f = open("sampleIds.py", "w")
f.write(ids)
f.close()